# State AI Policy: source validation

## Summary

We matched 18 sampled records to their LegiScan bills. Seven were in closed sessions, and the official histories support their existing `Failed - Adjourned` status.

Four records marked Pending have later signing or chaptering evidence. We record those observations separately; this notebook does not change the source data or modeling labels.

## Context and method

The sample includes six records each from California, Colorado and Texas. We first sought three Pending records per state, then filled the sample with terminal statuses using a fixed hash order. All six Texas records have terminal statuses. This convenience sample does not estimate a national error rate.

We kept the October 13 observations and their limitations, then checked bill and session identifiers, stages and history dates with LegiScan on October 19. Official histories and session-closure sources were also checked for the seven closed-session cases.

LegiScan stages such as `Introduced`, `Engrossed` and `Passed` are not the same as a final outcome. The notebook uses saved data and does not require an API key or internet connection.

## Data

### 1. Load the source and validation tables

Run from the repository root or `notebooks/`. `source_row` is the one-based row position in the original CSV; it is not a model feature.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display

project_root = Path.cwd()
if not (project_root / "data/raw/Final_Database.csv").exists():
    project_root = project_root.parent

raw_path = project_root / "data/raw/Final_Database.csv"
raw = pd.read_csv(raw_path, dtype=str, keep_default_na=False, skip_blank_lines=False)
validation_dir = project_root / "data/validation"
observations = pd.read_csv(validation_dir / "bill_status_observations.csv", dtype=str, keep_default_na=False)
comparison = pd.read_csv(validation_dir / "legiscan_comparison.csv", dtype=str, keep_default_na=False)
closed = pd.read_csv(validation_dir / "closed_session_review.csv", dtype=str, keep_default_na=False)

### 2. Check source records and bill identities

The checksum confirms the published CSV, and the source fields link each reviewed record to the correct row.

In [2]:
source_md5 = hashlib.md5(raw_path.read_bytes()).hexdigest()
assert source_md5 == "397a13c53e719e64e52725e7995fd8ca"
assert raw.shape == (2571, 29)
assert len(comparison) == 18 and comparison["source_row"].is_unique
assert comparison["source_md5"].eq(source_md5).all()
assert comparison["identity_matched"].eq("true").all()
assert comparison["api_bill_id"].is_unique

source_columns = ["Jurisdiction", "Bill_Number", "Bill_Title", "Introduction_Date", "Version_Date", "Bill_Status"]
positions = observations["source_row"].astype(int) - 1
pd.testing.assert_frame_equal(
    raw.loc[positions, source_columns].reset_index(drop=True),
    observations[source_columns].reset_index(drop=True),
)
assert comparison.set_index("source_row")["source_status"].to_dict() == observations.set_index("source_row")["Bill_Status"].to_dict()
display(comparison.groupby("state").agg(Records=("source_row", "size"), Matched=("identity_matched", lambda values: values.eq("true").sum())))

,Records,Matched
state,,
CA,6,6
CO,6,6
TX,6,6


## Results

### 3. Compare source statuses and API stages

The two fields describe different things and were checked on different dates. A different stage label alone does not mean the source outcome is wrong. The CSV includes bill and session identifiers, links and check times.

In [3]:
display(pd.crosstab(
    comparison["source_status"], comparison["api_stage"],
    rownames=["Source status"], colnames=["API stage"],
))

API stage,Engrossed,Failed,Introduced,Passed,Vetoed
Source status,,,,,
Enacted - Act No. 2025-341,0,0,0,1,0
Enacted - Act No. 2025-466,0,0,0,1,0
Enacted - Act No. 2025-673,0,0,0,1,0
Failed - Adjourned,2,1,5,0,0
Failed - Adjourned - Postponed Indefinitely,0,1,0,0,0
Pending,0,0,1,4,1


### 4. Review the seven closed-session cases

Colorado HB25B-1008 belongs to the 2025 extraordinary session, not the regular-session bill with the same number. The official page records Lost, and the session adjourned August 26, 2025.

The Texas 89th Regular Session ended June 2, 2025. The six histories show unfinished bills; the Legislative Reference Library says unfinished bills die at session end. We infer `Failed at session close` from those sources, rather than treating it as an API label.

All seven source records already say `Failed - Adjourned`, so we keep those statuses. A latest action date is not necessarily a failure date.

In [4]:
assert len(closed) == 7
assert closed["source_status"].eq("Failed - Adjourned").all()
assert set(closed["source_row"]) == set(comparison.loc[
    comparison["session_sine_die"].eq("1") & comparison["api_stage"].isin(["Introduced", "Engrossed", "Enrolled"]),
    "source_row",
])
display(closed[["state", "bill", "source_status", "api_stage", "last_action_date", "session_closed_date"]].rename(columns={
    "source_status": "Source status", "api_stage": "API stage",
    "last_action_date": "Last action", "session_closed_date": "Session closed",
}))

,state,bill,Source status,API stage,Last action,Session closed
0,CO,HB25B-1008,Failed - Adjourned,Introduced,2025-08-25,2025-08-26
1,TX,HB641,Failed - Adjourned,Introduced,2025-03-04,2025-06-02
2,TX,SB747,Failed - Adjourned,Engrossed,2025-05-26,2025-06-02
3,TX,SB1298,Failed - Adjourned,Introduced,2025-02-28,2025-06-02
4,TX,HB4448,Failed - Adjourned,Introduced,2025-04-02,2025-06-02
5,TX,SB1822,Failed - Adjourned,Engrossed,2025-05-07,2025-06-02
6,TX,HB1405,Failed - Adjourned,Introduced,2025-05-14,2025-06-02


### 5. Record four later enactment observations

Four source-Pending records have later signing or chaptering evidence. LegiScan labels code 4 as `Passed`; the history supplies the action details. These observations appeared in the October 13 check and were confirmed by the October 19 API histories.

Testing updated labels would require a clear cutoff. Later information should not be used as an input for a model claiming to predict an earlier outcome.

In [5]:
pending_with_evidence = comparison.loc[
    comparison["source_status"].eq("Pending") & comparison["signing_or_chaptering_evidence"].eq("True"),
    ["state", "bill", "source_status", "api_stage", "api_last_action_date", "api_last_action"],
]
assert len(pending_with_evidence) == 4
with pd.option_context("display.max_colwidth", 75):
    display(pending_with_evidence.rename(columns={"api_last_action_date": "Action date", "api_last_action": "Latest history action"}))

,state,bill,source_status,api_stage,Action date,Latest history action
0,CA,SB241,Pending,Passed,2025-10-01,"Chaptered by Secretary of State. Chapter 214, Statutes of 2025."
1,CA,SB951,Pending,Passed,2026-09-30,"Chaptered by Secretary of State. Chapter 860, Statutes of 2026."
7,CO,HB1195,Pending,Passed,2026-06-03,Governor Signed
8,CO,HB1263,Pending,Passed,2026-05-29,Governor Signed


### 6. Review unresolved cases and dates

- **California AB2291:** A canceled hearing does not establish a final outcome. The API reports `sine_die=0`, and we did not check a separate session calendar.
- **Colorado HB26-1210:** The API's Vetoed status agrees with the earlier veto history. The October 13 conflict with the official page's Signed Act/summary wording remains in `bill_status_observations.csv`.
- **Texas HB4448:** The source title differs from the official school-library caption, but the raw header identifies Hickland's HB4448. We have not established whether the full text matches the official version.

Three Texas introduction dates differ from the API's first-history dates, which reflect filing. We keep both dates rather than treating them as interchangeable.

In [6]:
source_dates = pd.to_datetime(observations["Introduction_Date"], format="%m/%d/%Y").dt.strftime("%Y-%m-%d")
date_review = comparison.merge(
    observations[["source_row", "Introduction_Date"]].assign(source_date_iso=source_dates),
    on="source_row", validate="one_to_one",
)
date_review = date_review.loc[date_review["source_date_iso"].ne(date_review["api_first_history_date"])]
display(date_review[["state", "bill", "Introduction_Date", "api_first_history_date"]].rename(columns={
    "Introduction_Date": "Source introduction", "api_first_history_date": "API first history",
}))

,state,bill,Source introduction,API first history
12,TX,HB641,01/14/2025,2024-11-12
13,TX,SB747,01/14/2025,2025-01-08
17,TX,HB1405,01/14/2025,2024-11-19


### 7. Compare 20 text-version references

`source_text_review.csv` links 20 selected records to cached version references checked October 20. The selection includes earlier outcome and sponsor cases and additional version checks; it is not random. The table includes source statuses and dates, bill and session identifiers, and links to introduced documents when available. Retrieval times remain in `text_version_review.csv`.

A header label and a date are separate pieces of evidence. For three Colorado bills, introduced dates come from official version tables because the API dates are placeholders. Delaware SCR18 has an undated draft, so its introduced date and document link remain blank. A version reference alone does not prove the source text is identical.

In [7]:
source_text_review = pd.read_csv(
    validation_dir / "source_text_review.csv", dtype=str, keep_default_na=False,
)
version_references = pd.read_csv(
    validation_dir / "text_version_review.csv", dtype=str, keep_default_na=False,
)
assert len(source_text_review) == 20 and source_text_review["source_row"].is_unique
assert source_text_review["source_md5"].eq(source_md5).all()
assert set(source_text_review["source_row"]) == set(version_references["source_row"])
identity_columns = ["api_bill_id", "api_session_id", "state", "bill_number"]
pd.testing.assert_frame_equal(
    source_text_review.set_index("source_row")[identity_columns].sort_index(),
    version_references.set_index("source_row")[identity_columns].sort_index(),
)
source_positions = source_text_review["source_row"].astype(int) - 1
assert source_positions.between(0, len(raw) - 1).all()
assert source_text_review["source_status"].tolist() == raw.loc[source_positions, "Bill_Status"].tolist()
assert source_text_review["source_version_date_raw"].tolist() == raw.loc[source_positions, "Version_Date"].tolist()
parsed_source_dates = pd.to_datetime(
    source_text_review["source_version_date_raw"], format="%m/%d/%Y",
).dt.strftime("%Y-%m-%d")
assert parsed_source_dates.equals(source_text_review["source_version_date_iso"])
assert source_text_review["full_text_identity_not_established"].eq("True").all()
source_text_review["later_stage_header"] = source_text_review["later_stage_header"].eq("True")

### 8. Screen raw bill-text headers

This screen checks all **2,571 source records** for explicit stage labels near the beginning of `Bill_Text`, before the digest, summary or bill body. It also recognizes California's formal bill-number label ending in “as introduced.” The phrase “Introduced by” identifies an author, not a version.

The screen finds 252 later-stage headers, 31 introduced headers and 2,288 without an explicit label. `NOT_EXPLICIT` means the stage is unknown. These labels cannot establish when the full text, summary or sponsor information became available, so the screen is a review aid rather than an introduction-time eligibility test.

In [8]:
import re


def header_stage(text):
    header = re.split(
        r"LEGISLATIVE COUNSEL'S DIGEST|Bill Summary|WHEREAS,|BE IT ENACTED",
        text, maxsplit=1, flags=re.I,
    )[0][:1800]
    lines = [line.strip() for line in header.splitlines() if line.strip()]
    for line in lines:
        if re.fullmatch(r"PREAMENDED|REENGROSSED|ENGROSSED|INTRODUCED|ENROLLED", line):
            return line
    if any(re.match(r"AMENDED IN (ASSEMBLY|SENATE)\b", line) for line in lines):
        return "AMENDED"
    if any(re.fullmatch(r"CHAPTER \d+", line) for line in lines):
        return "CHAPTERED"
    before_body = re.split(
        r"Bill Summary|WHEREAS,|BE IT ENACTED", text, maxsplit=1, flags=re.I,
    )[0][:1800]
    digest_label = re.search(
        r"LEGISLATIVE COUNSEL'S DIGEST\s+"
        r"(?:AB|SB|ACR|SCR|AJR|SJR|ACA|SCA|HR|SR)\s*\d+,\s+as introduced\b",
        before_body, flags=re.I,
    )
    if digest_label:
        return "INTRODUCED"
    return "NOT_EXPLICIT"


assert header_stage("A BILL\nBE IT ENACTED\nThis section quotes a bill as introduced.") == "NOT_EXPLICIT"
assert header_stage("INTRODUCED\nA BILL FOR AN ACT") == "INTRODUCED"


header_screen = pd.DataFrame({
    "source_row": range(1, len(raw) + 1),
    "screened_header_stage": raw["Bill_Text"].map(header_stage),
})
header_screen["source_row"] = header_screen["source_row"].astype(str)
header_counts = header_screen["screened_header_stage"].value_counts().rename_axis("Header stage")
display(header_counts.to_frame("Raw records"))

,Raw records
Header stage,
NOT_EXPLICIT,2288
CHAPTERED,101
AMENDED,99
INTRODUCED,31
ENROLLED,27
ENGROSSED,16
REENGROSSED,6
PREAMENDED,3


### 9. Review flagged records and dates

The 20 saved labels must match the raw-header screen. Nine sampled records have explicit later-stage labels: five grouped as Pending, three as Passed and one as Failed. The table also retains their exact source statuses. A flag does not by itself establish an incorrect outcome or target leakage.

Twelve source version dates are later than the introduced reference, seven match and one introduced date is unverified. Colorado HB26-1195 illustrates why dates alone are not enough: its date matches, but the source header says `PREAMENDED` while the official introduced PDF says `INTRODUCED`.

In [9]:
sample_screen = source_text_review.merge(header_screen, on="source_row", validate="one_to_one")
assert sample_screen["source_header_stage"].equals(sample_screen["screened_header_stage"])
later_stages = {"PREAMENDED", "REENGROSSED", "ENGROSSED", "AMENDED", "CHAPTERED", "ENROLLED"}
assert sample_screen["later_stage_header"].equals(sample_screen["screened_header_stage"].isin(later_stages))
flagged = sample_screen.loc[sample_screen["later_stage_header"]].copy()
flagged["source_grouped_status"] = raw.loc[
    flagged["source_row"].astype(int) - 1, "Bill_Status_Grouped",
].to_numpy()
assert len(flagged) == 9
assert flagged["source_grouped_status"].value_counts().to_dict() == {"Pending": 5, "Passed": 3, "Failed": 1}
with pd.option_context("display.max_colwidth", 55):
    display(flagged[[
        "source_row", "state", "bill_number", "source_status", "source_header_stage",
        "source_version_date_iso", "introduced_reference_date_iso",
    ]])

,source_row,state,bill_number,source_status,source_header_stage,source_version_date_iso,introduced_reference_date_iso
0,55,CA,SB241,Pending,AMENDED,2025-06-26,2025-01-30
8,1586,CA,SB951,Pending,AMENDED,2026-04-09,2026-02-02
9,1613,CA,AB2291,Pending,AMENDED,2026-04-06,2026-02-19
11,1628,CA,SB361,Enacted - Act No. 2025-466,CHAPTERED,2025-10-08,2025-02-13
12,1631,CA,AB578,Enacted - Act No. 2025-341,CHAPTERED,2025-10-06,2025-02-12
13,1632,CA,AB621,Enacted - Act No. 2025-673,CHAPTERED,2025-10-13,2025-02-13
14,1637,CO,HB1195,Pending,PREAMENDED,2026-02-11,2026-02-11
15,1638,CO,HB1210,Pending,REENGROSSED,2026-03-26,2026-02-13
17,1644,CO,HB1008,Failed - Adjourned,ENGROSSED,2025-08-24,2025-08-21


In [10]:
source_version_dates = pd.to_datetime(source_text_review["source_version_date_iso"], format="%Y-%m-%d")
introduced_dates = pd.to_datetime(
    source_text_review["introduced_reference_date_iso"], format="%Y-%m-%d", errors="coerce",
)
date_comparison = pd.Series("Introduced date unavailable", index=source_text_review.index)
date_comparison.loc[source_version_dates.gt(introduced_dates)] = "After introduced reference"
date_comparison.loc[source_version_dates.eq(introduced_dates)] = "Matches introduced reference"
date_comparison.loc[source_version_dates.lt(introduced_dates)] = "Before introduced reference"
display(date_comparison.value_counts().rename_axis("Source version date").to_frame("Sample records"))
assert date_comparison.value_counts().to_dict() == {
    "After introduced reference": 12,
    "Matches introduced reference": 7,
    "Introduced date unavailable": 1,
}
display(source_text_review.loc[source_text_review["source_row"].eq("1637"), [
    "state", "bill_number", "source_version_date_iso", "introduced_reference_date_iso",
    "source_header_stage", "introduced_document_url",
]])

,Sample records
Source version date,
After introduced reference,12
Matches introduced reference,7
Introduced date unavailable,1


,state,bill_number,source_version_date_iso,introduced_reference_date_iso,source_header_stage,introduced_document_url
14,CO,HB1195,2026-02-11,2026-02-11,PREAMENDED,https://leg.colorado.gov/bill_files/111810/dow...


Only three cached introduced PDFs were compared in full with the source text. Their normalized token sequences differ, including for the record labeled `INTRODUCED`. Formatting, extraction and content can all cause differences, so this check does not establish version identity. The remaining records have no full-text comparison.

We keep these findings separate from the modeling inputs. Removing the nine sampled flags would leave other later-stage headers; removing all detected later-stage headers would still leave 2,288 unknown headers and unverified summary and sponsor timing. Neither filter produces a verified introduction-time cohort. Comparisons of input variants should use the same rows and splits.

## Takeaways

Keep the source outcomes for the main retrospective analysis. The seven closed-session checks support their existing Failed/Adjourned labels; the four later enactment observations remain separate. The source does not establish a common observation date for all outcomes. A Pending label means the source recorded the outcome as unresolved.

The corpus contains mixed text versions. Removing the nine sampled flags would not create verified introduction-time data. Topic analysis can retain these bills, but using their summaries or topics for prediction requires separate timing and train/test checks.

Use the same rows and splits when comparing features. An updated-outcome experiment also needs a common observation date and a rule for unresolved cases.

## Sources

- Bhargava and Soliman, [U.S. State AI Legislation Corpus v4](https://zenodo.org/records/20781731), `Final_Database_10.csv`. The repository copy is unchanged.
- [LegiScan](https://legiscan.com/) API bill metadata/history, retrieved October 19, 2026. Selected fields are reorganized and renamed in `legiscan_comparison.csv`; stage labels follow the [API manual, Status/Progress section](https://api.legiscan.com/dl/LegiScan_API_User_Manual.pdf). LegiScan data is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).
- Official state bill-history links and per-record check times appear in the validation CSVs. The seven-bill review also cites the [Texas post-session FAQ](https://www.lrl.texas.gov/whatsNew/client/index.cfm/2025/6/3/Whats-Next-PostSession-FAQ-and-Bill-Statistics-89th-RS) and [Colorado extraordinary-session journal](https://content.leg.colorado.gov/sites/default/files/2025_house_journal.pdf), printed page 76 (PDF page 80).